# Using the Analytics Engine (AE) to reproduce annual consumption model
This notebook is an early draft attempt to reproduce the workflow CEC's Demand Forecast Unit takes to generate weather and climate information for the annual consumption model. Here the existing workflow is replicated, but connecting with new data from California's Fifth Climate Change Assessment.

To execute a given 'cell' of this notebook, place the cursor in the cell and press the 'play' icon, or simply press shift+enter together. Some cells will take longer to run, and you will see a [$\ast$] to the left of the cell while AE is still working.

## Step 0: Setup
This cell imports the python library [climakitae](https://github.com/cal-adapt/climakitae), our AE toolkit for climate data analysis, and any other specialized libraries needed for a given notebook.

In [ ]:
import numpy as np
from tqdm.auto import tqdm # Progress bar 
import pyproj
import rioxarray
import pandas as pd
import xarray as xr
import hvplot.xarray
import hvplot.pandas
import panel as pn
pn.extension()

In [ ]:
import climakitae as ck

Additionally, get set up to make the computing go faster by executing the following cell. It will likely take several minutes to spin up! Learn more about dask and see some common [troubleshooting tips on our FAQ page](https://analytics.cal-adapt.org/docs/faq/).

In [ ]:
from climakitae.cluster import Cluster
cluster = Cluster()
cluster.adapt(minimum=0, maximum=8)
client = cluster.get_client()
cluster

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

## Step 1: Retrieve the data
The data options can be set in two ways: 
1. using the app.select() GUI
2. hard-coding the selections by modifying the objects `app.selections` and `app.location`

Below, we hard-code in the selections instead of using the app.select GUI. We do this to improve reproducibility of the notebook and to align with demand forecast unit methodologies. But, feel free to run the commented out cell below if you'd like to select the data using the GUI, or preview the data options in our catalog. 

In [ ]:
# app.select()

### 1a) Data and location selections
Here, we'll set the resolution, timescale, and scenarios.

In [ ]:
%%capture
app.selections.scenario_historical = ['Historical Climate']
app.selections.scenario_ssp = ['SSP 3-7.0 -- Business as Usual']
app.selections.area_average = "No" 
app.selections.timescale = 'hourly'
app.selections.resolution = '9 km'
app.selections.time_slice = (2005, 2025) 

Next, we'll set the location subset.

In [ ]:
%%capture
app.location.area_subset = 'states'
app.location.cached_area = 'CA' # Just get data for the state of California

### 1b) Retrieve data for each variable of interest
Set your desired variables (and corresponding units) by modifying the values in the dictionary `variable_units_dict` below. Any variables in this list must match a variable that is already available in our data catalog. **You can preview the variables and available units in the app.select GUI.** 

In [ ]:
# Dictionary in the format {variable : units}
variable_units_dict = { 
    "Air Temperature at 2m" : "degF", 
    "Relative Humidity": "[0 to 100]", 
    "Dew point temperature": "degF"
} 

To retrieve the data for all variables, we will simply loop through the variable list. Then, we'll compile the data for each variable into a simple xarray Dataset object, allowing us to easily subset and modify all the variables at once. 

In [ ]:
def retrieve_dict_of_data(var_and_units, disable_pbar=False): 
    xr_list = [] # Empty list to store retrieved data for each variable 
    for variable, unit in tqdm(var_and_units.items(), disable=disable_pbar): 

        # Reset variable and units in app.selections 
        app.selections.variable = variable
        app.selections.units = unit

        # Retrieve the data 
        xr_ds = app.retrieve() 
        xr_list.append(xr_ds)

    # Merge data from all variables to form a single xr.Dataset object 
    data = xr.merge(xr_list) 
    return data

In [ ]:
data = retrieve_dict_of_data(variable_units_dict) 

### 1c) Preview the data 
You'll notice that each variable is a coordinate value in our output xarray dataset. This is xarray's way of managing multi-dimensional data. Since all the variables have the same spatial and temporal dimensions, they can be stored in the same object. To access an individual variable (for example, Air Temperature at 2m), you can simple type `data["Air Temperature at 2m"]` to get just data for that variable. 

In [ ]:
display(data) 

## Step 2: Get data from the closest grid cell to the weather station. 
As an example - to replicate the historical observations at Sacramento Executive Airport, grab the grid cell from the model nearest to the airport.

### 2a) Read in a csv file of the station coordinates 
We'll read it into a pandas DataFrame object.

In [ ]:
stations_df = pd.read_csv("CEC_Forecast_Weather Stations_California.csv", index_col="STATION")
stations_df.head(5) # Display the first 5 rows 

### 2b) Grab the closest grid cell to the weather station
To demonstrate this process, we'll use the Sacramento Executive Airport weather station.

In [ ]:
station_name = "SACRAMENTO EXECUTIVE AIRPORT"
one_station = stations_df.loc[station_name]

Next, we need to convert the lat/lon coordinate pair to the model's projection coordinates. We can easily do this using the python package pyproj by making a `Tranformer` object that will convert the coordinates for us.<br><br>Once we have the coordinates in the model projection, we can use [xarray's built in indexing function](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html), `.sel`, to easily grab the data from the grid cell that is closest to the weather station.

In [ ]:
def get_closest_gridcell(gridded_data, station_lon, station_lat): 
    # Make Transformer object 
    lat_lon_to_model_projection = pyproj.Transformer.from_crs(
        crs_from="epsg:4326", # Lat/lon
        crs_to=gridded_data.rio.crs, # Model projection
        always_xy=True
    )

    # Convert station coordinates 
    station_x, station_y = lat_lon_to_model_projection.transform(station_lon, station_lat) 

    # Get closest gridcell 
    closest_gridcell = gridded_data.sel(x=station_x, y=station_y, method='nearest')
    return closest_gridcell

In [ ]:
data_closest_gridcell = get_closest_gridcell(
    gridded_data=data,
    station_lon=one_station.LON_X, 
    station_lat=one_station.LAT_Y
)

### 2c) Load the data into memory 
This may take some time, because the data has to be loaded into memory and then subsetted to get the closest grid cell. All computations we've done before this step are actually computed in this step; before, we just see a preview of the data. 

In [ ]:
%%time
data_closest_gridcell = app.load(data_closest_gridcell)

Print the central coordinates of the nearest gridcell computed by xarray and compare to the actual station coordinates to observe how close they are.

In [ ]:
print("Weather station coords: (%.2f, %.2f)" % 
    (one_station.LAT_Y, one_station.LON_X) + "\n\
Nearest grid cell coords: (%.2f, %.2f)" % 
  (data_closest_gridcell.lat.item(), data_closest_gridcell.lon.item()))

### 2d) Output final data product as a csv file
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. In the output table, the first column is the time in units of UTC, and the second column are the various global climate models (which can be filtered in excel or in python code in the notebook). The other columns are the variables selected at the beginning of the notebook.

In [ ]:
data_closest_gridcell_df = data_closest_gridcell.isel(scenario=0).drop(
    ["x","y","landmask","lakemask","lat","lon","Lambert_Conformal","scenario"]
).to_dataframe()
data_closest_gridcell_df.head()

In [ ]:
filename = "hourly_data_closest_gridcell_{0}.csv".format(station_name.replace(" ", "_")).lower()
data_closest_gridcell_df.to_csv(filename, index=True)

## Step 3: Compute the median value of the grid cells in station's corresponding forecast zone
As an alternative to a single point, we can instead consider weather conditions across an entire forecast zone. In this example we calculate the median of all conditions across the Sacramento Municipal Utility District.

### 3a) Read in the shapefiles of the demand forecast zones 
We'll use this to find the demand forecast zone that contains the weather station, then find the overlapping grid cells over which to compute the median value. The geometries of each demand forecast zone is available in our data catalog. You can grab the data as a pandas DataFrame object using the code provided below, or subset by forecast zones easily in `app.select` in the location subsetting tab. 

In [ ]:
%%time
from climakitae.selectors import Boundaries
dfzs_df = Boundaries()._ca_forecast_zones # Load geometries from catalog
dfzs_df.head() # Display the first few rows  

### 3b) Crop the data to the corresponding forecast zone
We'll use [geopanda's `.contains` function](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html) to find the demand forecast zone where the weather station is located, and print the result. Then, we'll use [rioxarray](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.clip) to clip the data to the geometry that defines the forecast zone. 

In [ ]:
from shapely.geometry import Point

def clip_data_to_dfz(gridded_data, dfzs_df, station_lon, station_lat): 
    
    # Shapely Point object of the weather station 
    shapely_Point = Point(station_lon, station_lat) 

    # Get the name of the DFZ that contains that point 
    forecast_zone = dfzs_df.where(dfzs_df.contains(shapely_Point)).dropna()
    forecast_zone_name = forecast_zone.FZ_Name.item()
    print("Weather station: {0} \nDemand forecast zone: {1}".format(station_name.lower().title(),forecast_zone_name))
    
    # Clip data 
    clipped_data = gridded_data.rio.clip(
        geometries=[forecast_zone.geometry.item()], 
        crs=4326, 
        drop=True # Drop cells outside the forecast zone 
    )
    return clipped_data

In [ ]:
data_dfz = clip_data_to_dfz(
    gridded_data=data, 
    dfzs_df=dfzs_df, 
    station_lon=one_station.LON_X, 
    station_lat=one_station.LAT_Y
)

### 3c) Visualize both the Demand Forecast Zone and the weather station on the same map 

For simplicity's sake, we'll show just one variable and only two weeks of data. In the outputted map, you can see that our data contains multiple simulation options as well, which you can toggle between in the map's dropdown.

In [ ]:
# Used to add weather station as star to map 
point_df = pd.DataFrame({
    "longitude (degrees_east)":[one_station.LON_X],
    "latitude (degrees_north)":[one_station.LAT_Y],
    "weather station": station_name
})

# Grab subset of data and load into memory 
to_plot = data_dfz["Air Temperature at 2m"].isel(time = np.arange(0,13))
to_plot = app.load(to_plot)

In [ ]:
app.view(to_plot) * point_df.hvplot.points(
    hover_cols = ["weather station"], 
    marker = "star", size = 300, color = "black"
)

### 3d) Agreggate values across grid cells in the forecast zone 
**Chose your aggregation: median, mean, min, or max.** All can be easily computed by xarray with just one line of code, thanks to xarray. You could also write your own code to compute a weighted mean. 

In [ ]:
def aggregate_across_cells(gridded_data, aggregation="median"): 
    if aggregation == "median": 
        data_aggregated = gridded_data.median(dim=["x","y"])
    elif aggregation == "mean": 
        data_aggregated = gridded_data.mean(dim=["x","y"])
    elif dfz_aggregation == "min": 
        data_aggregated = gridded_data.min(dim=["x","y"])
    elif dfz_aggregation == "max": 
        data_aggregated = gridded_data.max(dim=["x","y"])
    # elif dfz_aggregation == "weighted mean": ...
    return data_aggregated

Set your aggregation of choice and perform the aggregation using the function defined above.

In [ ]:
aggregation = "median" # Set your aggregation choice
data_dfz_aggregated = aggregate_across_cells(data_dfz, aggregation)

Lastly, let's load this final data product into memory 

In [ ]:
%%time
data_dfz_aggregated = app.load(data_dfz_aggregated)

### 3e) Output final data product as a csv file
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. 

In [ ]:
dfz_aggregated_df = data_dfz_aggregated.isel(scenario=0).drop(
    ["scenario","Lambert_Conformal"]).to_dataframe()
dfz_aggregated_df.head()

In [ ]:
filename = "dfz_aggregated_{0}_{1}.csv".format(aggregation, station_name.replace(" ", "_").lower())
dfz_aggregated_df.to_csv(filename, index=True)

## Step 4: Compute heating degree days and cooling degree days
Here, a heating degree day (HDD) is calculated by computing how many degrees Farenheit **colder** the daily temperature is from a standard temperature of 65 degrees Farenheit. A cooling degree day (CDD) is calulcated by computing how many degrees **warmer** the daily temperature is from the same standard temperature.

### 4a) Decide which input data you want to use 
You can use the closest grid cell to the weather station, which we computed in step 3. Or, you can use the data agreggated over the demand forecast zone, which we computed in step 4. Just comment out whichever you don't want to use. We've chosen to show the analysis with the agreggated DFZ data, but if you want to use the closest grid cell data, just comment out the DFZ cells and uncomment out the closest grid cells.<br><br>Depending on the input data, we will also set a new variable defining the number of grid cells. This will of course be just 1 for the closest grid cell method; for the agreggated DFZ data, however, this value will change depending on the size of the DFZ. This information is used to compute the annual agreggate HDD and CDD in step 5c.

In [ ]:
# CLOSEST GRID CELL 
# data_to_use = data_closest_gridcell
# num_grid_cells = 1

# AGGREGATED CELLS IN DFZ 
data_to_use = data_dfz_aggregated 
num_grid_cells = data_dfz.x.size * data_dfz.y.size # Number of grid cells within the demand forecast region

### 4b) Compute HDD and CDD 
HDD = 65 - temperature<br>
CDD = (-1)\*(65 - temperature)<br><br>
For HDD, we can just subtract the 2m temperature from 65 degrees Farenheight, then set any negative to 0. For CDD, we will do the same, but will then multiply by -1 to turn negative values to positive, then set negative values to 0. We need to multiply by -1 for CDD to avoid having all negative values; for example, on a day of 80F, CDD = 65 - 80 = -15, but the CDD value is +15. Multiplying -15 by -1 will give us the true value of 15. 

In [ ]:
def compute_hdd_cdd(t2, standard_temp = 65): 
    """Compute heating degree days (HDD) and cooling degree days (CDD) 
    
    Args: 
        t2 (xr.DataArray): Air temperature at 2m gridded data 
        standard_temp (int, optional): standard temperature in Fahrenheit (default to 65)
        
    Returns: 
        hdd, cdd (xr.DataArray) 
    """
    # Subtract t2 from the standard reference temperature 
    deg_less_than_standard = 65 - t2

    # Compute HDD: Find positive difference (i.e. days < 65 degF) 
    hdd = deg_less_than_standard.where(deg_less_than_standard > 0, 0) # Replace negative values with 0
    hdd.name = "Heating Degree Days" 

    # Compute CDD: Find negative difference (i.e. days > 65 degF)
    cdd = (-1)*deg_less_than_standard.where(deg_less_than_standard < 0, 0) # Replace positive values with 0
    cdd.name = "Cooling Degree Days" 
    
    return (hdd, cdd) 

In [ ]:
t2 = data_to_use["Air Temperature at 2m"]
hdd, cdd = compute_hdd_cdd(t2, standard_temp = 65)

### 4c) Aggregate annually to find HDD and CDD per year
To do this, we will first group the data by year and compute a sum across space and time. Then, we will divide the annual aggregated data by the number of grid cells over which the sum was computed. 

In [ ]:
def compute_annual_aggreggate(data, name, num_grid_cells):  
    annual_ag = data.squeeze().groupby('time.year').sum(['time']) # Aggregate annually
    annual_ag = annual_ag/num_grid_cells # Divide by number of gridcells 
    annual_ag.name = name # Give new name to dataset
    return annual_ag

In [ ]:
hdd_annual = compute_annual_aggreggate(
    data=hdd, 
    name="Annual Heating Degree Days (HDD)", 
    num_grid_cells=num_grid_cells
)
cdd_annual = compute_annual_aggreggate(
    data=cdd, 
    name="Annual Cooling Degree Days (CDD)", 
    num_grid_cells=num_grid_cells
)

### 4d) Compute the multimodel mean, min, and max. 
We'll add these to the simulation coordinate in our main datasets, `hdd_annual` and `cdd_annual`, so they can be easily accessed for plotting.

In [ ]:
def compute_multimodel_stats(hdd_annual, cdd_annual): 
    # Compute mean across simulation dimensions and add is as a coordinate
    hdd_sim_mean = hdd_annual.mean(dim = "simulation").assign_coords({"simulation":"simulation mean"}).expand_dims("simulation") 
    cdd_sim_mean = cdd_annual.mean(dim = "simulation").assign_coords({"simulation":"simulation mean"}).expand_dims("simulation") 

    # Compute multimodel min 
    hdd_sim_min = hdd_annual.min(dim = "simulation").assign_coords({"simulation":"simulation min"}).expand_dims("simulation") 
    cdd_sim_min = cdd_annual.min(dim = "simulation").assign_coords({"simulation":"simulation min"}).expand_dims("simulation") 

    # Compute multimodel max 
    hdd_sim_max = hdd_annual.max(dim = "simulation").assign_coords({"simulation":"simulation max"}).expand_dims("simulation") 
    cdd_sim_max = cdd_annual.max(dim = "simulation").assign_coords({"simulation":"simulation max"}).expand_dims("simulation") 

    # Add to main dataset
    hdd_concat = xr.concat([hdd_annual, hdd_sim_mean, hdd_sim_min, hdd_sim_max], dim = "simulation")
    cdd_concat =  xr.concat([cdd_annual, cdd_sim_mean, cdd_sim_min, cdd_sim_max], dim = "simulation")
    return hdd_concat, cdd_concat

In [ ]:
hdd_annual, cdd_annual = compute_multimodel_stats(hdd_annual, cdd_annual)

### 4e) Compute a trendline using the mean of all simulations
We'll find the coefficients for a first degree (linear) polynomial using [numpy's `polyfit` function](https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html). The returned coefficients (**m** and **b** in the code below) will allow us to compute the trendline using the linear polynomial y = mx + b, where **y** is the trendline and **x** is the years. 

In [ ]:
def trendline(data): 
    data_sim_mean = data.sel(simulation="simulation mean") 
    m, b = data_sim_mean.polyfit(dim="year", deg=1).polyfit_coefficients.values
    trendline = m*data_sim_mean.year + b # y = mx + b 
    trendline.name = "trendline" 
    return trendline

In [ ]:
%%time
hdd_trendline = trendline(hdd_annual) 
cdd_trendline = trendline(cdd_annual) 

### 4f) Visualize the results
Using the python package *hvplot*, we can easily make a line plot of the annual aggregated data. To do this, we'll plot the annual HDD, then add the trendline on top. The code to generate the plot is contained in a function `plot_data` defined below. 

In [ ]:
def plot_data(annual_data, trendline, title="title"): 
    return annual_data.hvplot.line(
        x="year", by="simulation", 
        width=800, height=350,
        title=title,
        yformatter='%.0f' # Remove scientific notation
    ) * trendline.hvplot.line(  # Add trendline
        x="year", 
        color="black", 
        line_dash='dashed', 
        label="trendline"
    ) 

In [ ]:
plot_data(
    annual_data = hdd_annual, 
    trendline = hdd_trendline, 
    title = "Annual Aggregate Heating Degree Days"
)

In [ ]:
plot_data(
    annual_data = cdd_annual, 
    trendline = cdd_trendline, 
    title = "Annual Aggregate Cooling Degree Days"
)

### 4g) Output data as csv files
We'll drop all unneeded coordinates and convert our xarray Dataset to a pandas Dataframe, allowing us to easily output the final data product to a csv file. 

In [ ]:
# Merge and simplify data 
hdd_cdd_combined = xr.merge([hdd_annual, cdd_annual]).drop(["Lambert_Conformal","scenario"])
hdd_cdd_combined = app.load(hdd_cdd_combined) 

# Convert to pandas dataframe 
hdd_cdd_df = hdd_cdd_combined.to_dataframe()
hdd_cdd_df.head()

In [ ]:
filename = "annual_hdd_cdd_{0}.csv".format(station_name.replace(" ", "_").lower())
hdd_cdd_df.to_csv(filename, index=True)

## Step 5: Using the same methods, find the daily min, max, mean temperature at target grid cells
We can do this easily by recycling the functions used in steps 3 and 4. 

### 5a) Change the timescale to daily
To do this, we need to modify the `app.selections` object. 

In [ ]:
app.selections.timescale = 'daily'

### 5b) Retrieve data for the mean, min, and max 2 meter air temperature
We'll do this by using the function `retrieve_dict_of_data` defined in step 2 of this notebook.

In [ ]:
daily_data = retrieve_dict_of_data({
    "Air Temperature at 2m" : "degF", 
    "Daily minimum air temperature at 2m": "degF", 
    "Daily maximum air temperature at 2m": "degF"
}) 

### 5c) Decide if you want to use the closest grid cell method, or the aggregated DFZ method. 
Set your method of choice below.

In [ ]:
#method = "closest grid cell" 
method = "aggregated DFZ" 

Next, either find the closest grid cell, or aggregate the data across the corresponding DFZ zone, depending on how the variable `method` is set above.

In [ ]:
# Clip data to get just the single closest grid cell
if method == "closest grid cell": 
    daily_data_clipped = get_closest_gridcell(daily_data, one_station.LON_X, one_station.LAT_Y)

# Clip data to the demand forecast zone, aggregate across all grid cells 
elif method == "aggregated DFZ": 
    daily_data_dfz = clip_data_to_dfz(daily_data, dfzs_df, one_station.LON_X, one_station.LAT_Y)
    daily_data_clipped = aggregate_across_cells(daily_data_dfz, "median")

### 5d) Output the data to csv files
First, we'll load the data into memory. Then, we will convert the xarray dataset to a pandas dataframe object. Lastly, we will output the dataframe to a csv file. 

In [ ]:
%%time
daily_data_clipped = app.load(daily_data_clipped)

In [ ]:
daily_df = daily_data_clipped.isel(scenario=0).drop(["Lambert_Conformal","scenario"]).to_dataframe()
daily_df.head()

In [ ]:
filename = "daily_2m_temp_{0}.csv".format(station_name.replace(" ", "_").lower())
daily_df.to_csv(filename, index=True)

## Step 6: Perform the same analyses for all weather stations
We'll loop through each weather station and perform the same analyses as above: (1) hourly data and (2) annual aggreggate HDD/CDD)

### 6a) Select your input data and aggregation type 
Do you want to continue with this notebook using the single grid cell closest to the weather station, or the median value over the forecast zone? To set your input data, simply change the variable `method` to "aggregated dfz" or "closest grid cell". 

Also, you'll need to chose the type of aggregation you want to perform within the DFZ. 

In [ ]:
#method = "closest grid cell" 
method = "aggregated DFZ" 
aggregation = "median"

### 6b) Run the analysis 
Everything from the notebook has been condensed into several helper functions, which will be run together to perform the workflow for each weather station. 

First, reset app.selections and app.location to desired values 

In [ ]:
# app.selections.scenario_historical = ['Historical Climate']
# app.selections.scenario_ssp = ['SSP 3-7.0 -- Business as Usual']
# app.selections.area_average = False 
# app.selections.timescale = 'hourly'
# app.selections.resolution = '9 km'
# app.selections.time_slice = (2005, 2025) 
# app.location.area_subset = 'states'
# app.location.cached_area = 'CA'

Next, loop through the weather stations and perform the analysis for each station.

In [ ]:
# print("Method selected: {0}".format(method))

# for i in tqdm(range(len(stations_df.index))):
#     # Get coordinate information from the station
#     station_name = stations_df.index[i]
#     print("Running analysis for {0}".format(station_name))
#     one_station = stations_df.loc[station_name]
    
#     # Retrieve the data 
#     print("Reading data for each variable...")
#     data = retrieve_dict_of_data({ 
#         "Air Temperature at 2m" : "degF", 
#         "Relative Humidity": "[0 to 100]", 
#         "Dew point temperature": "degF"
#     }, disable_pbar=True)
#     print("Complete!") 

#     # Clip data to get just the single closest grid cell
#     if method == "closest grid cell": 
#         data_closest_gridcell = get_closest_gridcell(data, one_station.LON_X, one_station.LAT_Y)
#         data_closest_gridcell = app.load(data_closest_gridcell) 
#         data_to_use = data_closest_gridcell
#         num_grid_cells = 1
        
#         # Output data as csv 
#         filename = "hourly_data_closest_gridcell_{0}.csv".format(station_name.replace(" ", "_")).lower()  
#         data_closest_gridcell_df = data_closest_gridcell.isel(scenario=0).drop(
#             ["x","y","landmask","lakemask","lat","lon","Lambert_Conformal","scenario"]
#         ).to_dataframe()
#         data_closest_gridcell_df.to_csv(filename, index=True)

#     # Clip data to the demand forecast zone, aggregate across all grid cells 
#     elif method == "aggregated DFZ": 
#         data_dfz = clip_data_to_dfz(data, dfzs_df, one_station.LON_X, one_station.LAT_Y)
#         data_dfz_aggregated = aggregate_across_cells(data_dfz, aggregation)
#         data_dfz_aggregated = app.load(data_dfz_aggregated) 
#         data_to_use = data_dfz_aggregated 
#         num_grid_cells = data_dfz.x.size * data_dfz.y.size 
        
#         # Output data 
#         filename = "dfz_aggregated_{0}_{1}.csv".format(aggregation, station_name.replace(" ", "_").lower())
#         dfz_aggregated_df = data_dfz_aggregated.isel(scenario=0).drop(
#             ["scenario","Lambert_Conformal"]).to_dataframe()
#         dfz_aggregated_df.to_csv(
#             filename, 
#             index=True
#         )
        
#     # Compute HDD and CDD 
#     t2 = data_to_use["Air Temperature at 2m"]
#     hdd, cdd = compute_hdd_cdd(t2, standard_temp=65)
#     hdd_annual = compute_annual_aggreggate(hdd, "Annual Heating Degree Days (HDD)", num_grid_cells)
#     cdd_annual = compute_annual_aggreggate(cdd, "Annual Cooling Degree Days (CDD)", num_grid_cells)
#     hdd_annual, cdd_annual = compute_multimodel_stats(hdd_annual, cdd_annual)
#     hdd_cdd_combined = xr.merge([hdd_annual, cdd_annual]).drop(["Lambert_Conformal","scenario"])
#     hdd_cdd_combined = app.load(hdd_cdd_combined)
    
#     # Output data 
#     filename = "annual_hdd_cdd_{0}.csv".format(station_name.replace(" ", "_").lower())
#     hdd_cdd_df = hdd_cdd_combined.to_dataframe()
#     hdd_cdd_df.to_csv(filename, index=True)
#     print("Completed analysis for {0}".format(station_name))

## Step 7: Close the compute cluster
Lastly, when you are done, close your cluster resources to free them up for the next time you work. 

In [ ]:
client.close()